In [1]:
%pip install pandas openpyxl


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

from slot_util import (
    load_mega_symbol_tables,
    print_df_attrs,
    get_reel_symbols,
    get_reelstop_df,
    add_wild_combination_counts,
    get_line_with_wild_pattern_df,
    get_mega_symbol_state_df,
    get_mega_symbol_line_probability_df,
    get_mega_pattern_probability_df,
)

In [3]:
(
    reel_df,
    paytable_df,
    payline_df,
    mega_case_df,
    mega_symbol_df,
) = load_mega_symbol_tables("sheet/MegaSymbol.xlsx")

In [4]:
for df in [
    reel_df,
    paytable_df,
    payline_df,
    mega_case_df,
    mega_symbol_df,
]:
    print_df_attrs(df)
    display(df)

shape: (20, 5)
表: 滾輪表
列: Stop 索引
行: 第幾個滾輪


,R1,R2,R3,R4,R5
0,W,W,W,W,W
1,K,K,K,K,K
2,Q,Q,Q,Q,Q
3,J,J,J,J,J
4,10,10,10,10,10
5,9,9,9,9,9
6,K,9,9,9,9
7,Q,K,K,K,K
8,J,Q,Q,Q,Q
9,10,J,J,J,J


shape: (6, 3)
表: 賠率表
列: 連線的 Symbol
行: 由左到右的連線數


,3,4,5
Symbol,,,
W,150,300,1000
K,5,30,100
Q,5,25,100
J,5,20,75
10,5,20,75
9,5,10,50


shape: (5, 5)
表: 賠線表
列: Line
行: 各滾輪上的 Row 索引


,R1,R2,R3,R4,R5
Line,,,,,
1,0,0,0,0,0
2,1,1,1,1,1
3,2,2,2,2,2
4,0,1,2,1,0
5,2,1,0,1,2


shape: (6, 4)
表: Mega Symbol 覆蓋 Case
列: Case
行.Prob: Case 出現機率
行.Top/Middle/Bottom: 各 Row 是否被覆蓋


,Prob,Top,Middle,Bottom
Case,,,,
1,0.70,0,0,0
2,0.05,1,0,0
3,0.05,0,0,1
4,0.08,1,1,0
5,0.08,0,1,1
6,0.04,1,1,1


shape: (6, 1)
表: Mega Symbol 機率表
列: Mega Symbol
行.Prob: Mega Symbol 出現機率


,Prob
Symbol,
W,0.166667
K,0.166667
Q,0.166667
J,0.166667
10,0.166667
9,0.166667


In [5]:
symbols = get_reel_symbols(reel_df)
symbols

['W', 'K', 'Q', 'J', '10', '9']

In [6]:
reelstop_df = get_reelstop_df(reel_df, symbols)

In [7]:
add_wild_combination_counts(reelstop_df, symbols)

In [8]:
print_df_attrs(reelstop_df)
display(reelstop_df)

shape: (17, 5)
表: Symbol 在各滾輪的 Stop 數
列: Symbol
行: 第幾個滾輪


,R1,R2,R3,R4,R5
Symbol,,,,,
W,1,1,1,1,1
K,4,3,3,3,3
Q,4,3,3,3,3
J,4,4,4,4,4
10,4,4,4,4,4
9,3,5,5,5,5
-,20,20,20,20,20
WK,5,4,4,4,4
WQ,5,4,4,4,4


In [9]:
mega_state_df = get_mega_symbol_state_df(mega_symbol_df)

In [10]:
print_df_attrs(mega_state_df)
display(mega_state_df)

shape: (48, 6)
表: Mega Symbol State
列: MegaSymbol x Pattern
行.State: MegaSymbol_Pattern
行.Pattern: R2、R3、R4 是否被 Mega Symbol 覆蓋


,State,MegaSymbol,Pattern,R2,R3,R4
0,W_000,W,000,0,0,0
1,W_001,W,001,0,0,1
2,W_010,W,010,0,1,0
3,W_011,W,011,0,1,1
4,W_100,W,100,1,0,0
5,W_101,W,101,1,0,1
6,W_110,W,110,1,1,0
7,W_111,W,111,1,1,1
8,K_000,K,000,0,0,0
9,K_001,K,001,0,0,1


In [11]:
line_df = get_line_with_wild_pattern_df(
    reel_df,
    paytable_df,
    symbols,
)
mega_line_df = get_mega_symbol_line_probability_df(
    line_df,
    mega_state_df,
    reelstop_df,
    paytable_df,
)

In [12]:
print_df_attrs(mega_line_df)
display(mega_line_df)

shape: (268, 58)
表: 各 Mega Symbol State 的連線組合機率
列: LineType x Symbol x MatchCount
行.Pay: 該連線組合的得分
行.State: P(Line combination | Mega Symbol state)


,LineType,Symbol,MatchCount,LeadingWCount,R1,R2,R3,R4,R5,Pay,...,10_110,10_111,9_000,9_001,9_010,9_011,9_100,9_101,9_110,9_111
0,LineK,K,5,0,K,K,K,K,K,100,...,0.0,0.0,1.012500e-04,0.0000,0.0,0.0,0.00000,0.0,0.0,0.0
1,LineK,K,4,0,K,K,K,K,X_WK,30,...,0.0,0.0,5.400000e-04,0.0000,0.0,0.0,0.00000,0.0,0.0,0.0
2,LineK,K,3,0,K,K,K,X_WK,-,5,...,0.0,0.0,3.600000e-03,0.0045,0.0,0.0,0.00000,0.0,0.0,0.0
3,LineK,Q,5,0,Q,Q,Q,Q,Q,100,...,0.0,0.0,1.012500e-04,0.0000,0.0,0.0,0.00000,0.0,0.0,0.0
4,LineK,Q,4,0,Q,Q,Q,Q,X_WQ,25,...,0.0,0.0,5.400000e-04,0.0000,0.0,0.0,0.00000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263,LineWK,9,3,1,W,9,W,X_W9,-,5,...,0.0,0.0,4.375000e-04,0.0000,0.0,0.0,0.00175,0.0,0.0,0.0
264,LineWK,9,3,0,9,W,W,X_W9,-,5,...,0.0,0.0,2.625000e-04,0.0000,0.0,0.0,0.00000,0.0,0.0,0.0
265,LineW,W,5,5,W,W,W,W,W,1000,...,0.0,0.0,3.125000e-07,0.0000,0.0,0.0,0.00000,0.0,0.0,0.0
266,LineW,W,4,4,W,W,W,W,S,300,...,0.0,0.0,0.000000e+00,0.0000,0.0,0.0,0.00000,0.0,0.0,0.0


In [13]:
state_cols = mega_state_df["State"].tolist()
mega_state_rtp = (
    mega_line_df[state_cols]
    .mul(mega_line_df["Pay"], axis="index")
    .sum(axis="index")
    .rename("RTP")
)

In [14]:
display(
    mega_state_rtp
    .to_frame()
    .style.format({"RTP": "{:.2%}"})
)

,RTP
W_000,82.32%
W_001,231.09%
W_010,361.94%
W_011,1014.25%
W_100,361.94%
W_101,1014.25%
W_110,1931.25%
W_111,5035.00%
K_000,82.32%
K_001,74.70%


In [15]:
mega_pattern_probability_df = get_mega_pattern_probability_df(
    payline_df,
    mega_case_df,
)

In [16]:
print_df_attrs(mega_pattern_probability_df)
display(mega_pattern_probability_df.style.format("{:.2%}"))

shape: (5, 8)
表: 各 Payline 的 Mega Symbol Pattern 機率
列: Line
行.Pattern: P(該 Line 的 R2、R3、R4 覆蓋 Pattern)


Pattern,000,001,010,011,100,101,110,111
Line,,,,,,,,
1,83.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,17.00%
2,80.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,20.00%
3,83.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,17.00%
4,75.00%,0.00%,5.00%,0.00%,0.00%,8.00%,0.00%,12.00%
5,75.00%,0.00%,5.00%,0.00%,0.00%,8.00%,0.00%,12.00%


In [17]:
mega_state_probability_df = pd.DataFrame({
    state.State: (
        mega_pattern_probability_df[state.Pattern]
        * mega_symbol_df.loc[state.MegaSymbol, "Prob"]
    )
    for state in mega_state_df.itertuples(index=False)
})
mega_state_probability_df.index.name = "Line"

In [18]:
display(mega_state_probability_df.style.format("{:.2%}"))

,W_000,W_001,W_010,W_011,W_100,W_101,W_110,W_111,K_000,K_001,K_010,K_011,K_100,K_101,K_110,K_111,Q_000,Q_001,Q_010,Q_011,Q_100,Q_101,Q_110,Q_111,J_000,J_001,J_010,J_011,J_100,J_101,J_110,J_111,10_000,10_001,10_010,10_011,10_100,10_101,10_110,10_111,9_000,9_001,9_010,9_011,9_100,9_101,9_110,9_111
Line,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,13.83%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,2.83%,13.83%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,2.83%,13.83%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,2.83%,13.83%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,2.83%,13.83%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,2.83%,13.83%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,2.83%
2,13.33%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,3.33%,13.33%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,3.33%,13.33%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,3.33%,13.33%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,3.33%,13.33%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,3.33%,13.33%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,3.33%
3,13.83%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,2.83%,13.83%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,2.83%,13.83%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,2.83%,13.83%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,2.83%,13.83%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,2.83%,13.83%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,2.83%
4,12.50%,0.00%,0.83%,0.00%,0.00%,1.33%,0.00%,2.00%,12.50%,0.00%,0.83%,0.00%,0.00%,1.33%,0.00%,2.00%,12.50%,0.00%,0.83%,0.00%,0.00%,1.33%,0.00%,2.00%,12.50%,0.00%,0.83%,0.00%,0.00%,1.33%,0.00%,2.00%,12.50%,0.00%,0.83%,0.00%,0.00%,1.33%,0.00%,2.00%,12.50%,0.00%,0.83%,0.00%,0.00%,1.33%,0.00%,2.00%
5,12.50%,0.00%,0.83%,0.00%,0.00%,1.33%,0.00%,2.00%,12.50%,0.00%,0.83%,0.00%,0.00%,1.33%,0.00%,2.00%,12.50%,0.00%,0.83%,0.00%,0.00%,1.33%,0.00%,2.00%,12.50%,0.00%,0.83%,0.00%,0.00%,1.33%,0.00%,2.00%,12.50%,0.00%,0.83%,0.00%,0.00%,1.33%,0.00%,2.00%,12.50%,0.00%,0.83%,0.00%,0.00%,1.33%,0.00%,2.00%


In [19]:
mega_state_contribution_df = (
    mega_state_probability_df.mul(
        mega_state_rtp,
        axis="columns",
    )
)
mega_state_contribution_df["RTP"] = (
    mega_state_contribution_df.sum(axis="columns")
)

In [20]:
display(mega_state_contribution_df.style.format("{:.2%}"))

,W_000,W_001,W_010,W_011,W_100,W_101,W_110,W_111,K_000,K_001,K_010,K_011,K_100,K_101,K_110,K_111,Q_000,Q_001,Q_010,Q_011,Q_100,Q_101,Q_110,Q_111,J_000,J_001,J_010,J_011,J_100,J_101,J_110,J_111,10_000,10_001,10_010,10_011,10_100,10_101,10_110,10_111,9_000,9_001,9_010,9_011,9_100,9_101,9_110,9_111,RTP
Line,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,11.39%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,142.66%,11.39%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,31.17%,11.39%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,28.33%,11.39%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,23.91%,11.39%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,23.91%,11.39%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,12.47%,330.76%
2,10.98%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,167.83%,10.98%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,36.67%,10.98%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,33.33%,10.98%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,28.13%,10.98%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,28.13%,10.98%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,14.67%,374.60%
3,11.39%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,142.66%,11.39%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,31.17%,11.39%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,28.33%,11.39%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,23.91%,11.39%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,23.91%,11.39%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,12.47%,330.76%
4,10.29%,0.00%,3.02%,0.00%,0.00%,13.52%,0.00%,100.70%,10.29%,0.00%,0.53%,0.00%,0.00%,2.93%,0.00%,22.00%,10.29%,0.00%,0.50%,0.00%,0.00%,2.67%,0.00%,20.00%,10.29%,0.00%,0.63%,0.00%,0.00%,2.81%,0.00%,16.88%,10.29%,0.00%,0.63%,0.00%,0.00%,2.81%,0.00%,16.88%,10.29%,0.00%,0.51%,0.00%,0.00%,1.76%,0.00%,8.80%,279.32%
5,10.29%,0.00%,3.02%,0.00%,0.00%,13.52%,0.00%,100.70%,10.29%,0.00%,0.53%,0.00%,0.00%,2.93%,0.00%,22.00%,10.29%,0.00%,0.50%,0.00%,0.00%,2.67%,0.00%,20.00%,10.29%,0.00%,0.63%,0.00%,0.00%,2.81%,0.00%,16.88%,10.29%,0.00%,0.63%,0.00%,0.00%,2.81%,0.00%,16.88%,10.29%,0.00%,0.51%,0.00%,0.00%,1.76%,0.00%,8.80%,279.32%


In [21]:
mega_total_rtp = mega_state_contribution_df["RTP"].mean()

In [22]:
print(f"RTP: {mega_total_rtp:.2%}")

RTP: 318.95%
